# 1. Setup and Libraries

In [19]:
import os
import glob
import json
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import time
import scipy.signal

# 2. Dataset Directories

In [20]:
dataset_Folder = "G:/My Drive/GP Datasets/fit3d_data/"
file_path = dataset_Folder + 'fit3d_info.json'

# 3. Load FIT3D Metadata

In [21]:
info_JSON = pd.read_json(file_path, typ='series')
print("JSON loaded successfully as a series:")
print(info_JSON.head())

val_subj_names = ['s03', 's11']
all_train = [sub for sub in info_JSON['train_subj_names'] if sub not in val_subj_names]
all_camera_names = info_JSON['all_camera_names']

# Extract all actions from all subjects and find unique values
all_actions_list = [action for actions in info_JSON['subj_to_act'].values() for action in actions]
unique_actions = sorted(list(set(all_actions_list)))

print(f"Number of unique actions: {len(unique_actions)}")
print("Unique actions:")
print(unique_actions)

JSON loaded successfully as a series:
subj_to_act         {'s03': ['band_pull_apart', 'dumbbell_high_pul...
test_subj_names                                       [s02, s12, s13]
train_subj_names             [s03, s04, s05, s07, s08, s09, s10, s11]
all_camera_names             [50591643, 58860488, 60457274, 65906101]
dtype: object
Number of unique actions: 47
Unique actions:
['band_pull_apart', 'barbell_dead_row', 'barbell_row', 'barbell_shrug', 'burpees', 'clean_and_press', 'deadlift', 'diamond_pushup', 'drag_curl', 'dumbbell_biceps_curls', 'dumbbell_curl_trifecta', 'dumbbell_hammer_curls', 'dumbbell_high_pulls', 'dumbbell_overhead_shoulder_press', 'dumbbell_reverse_lunge', 'dumbbell_scaptions', 'man_maker', 'mule_kick', 'neutral_overhead_shoulder_press', 'one_arm_row', 'overhead_extension_thruster', 'overhead_trap_raises', 'pushup', 'side_lateral_raise', 'squat', 'standing_ab_twists', 'w_raise', 'walk_the_box', 'warmup_1', 'warmup_10', 'warmup_11', 'warmup_12', 'warmup_13', 'warmup_14

# 10. Repetition Segmentation & Counting Evaluation
Evaluates the repetition-detection pipeline using **joints3d_25** 3D skeletons 
from the Fit3D dataset as input.  
Ground truth is sourced from `rep_ann.json` which stores a list of frame-boundary 
indices per action.

**Metrics:**
- **Repetition Segmentation:** Intersection-over-Union (IoU) between predicted and 
annotated repetition intervals (averaged over repetitions)
- **Repetition Counting:** Off-By-One error rate (OBO) and Mean Absolute Error (MAE) of counts

## 10.1 FIT3D Joint Mapping & Exercise Name Mapping

In [22]:
import sys
sys.path.append('d:/GP/Pose/Code')
from exercise_config import EXERCISE_TO_CONFIG, JOINT_DEFINITIONS

FIT3D_JOINT_MAP = {
    'hip':        (1, 4),
    'knee':       (2, 5),
    'ankle':      (3, 6),
    'shoulder':   (11, 14),
    'elbow':      (12, 15),
    'wrist':      (13, 16),
    'index':      (18, 21),
    'nose':       None,
    'vertical':   None,
    'horizontal': None,
}

FIT3D_TO_CONFIG_NAME = {
    # Strictly single-phase, non-alternating exercises
    'squat':                           'squat',
    'deadlift':                        'deadlift',
    'pushup':                          'push-up',
    'diamond_pushup':                  'push-up',
    'dumbbell_biceps_curls':           'barbell biceps curl',
    'dumbbell_hammer_curls':           'hammer curl',
    'dumbbell_overhead_shoulder_press':'shoulder press',
    'neutral_overhead_shoulder_press': 'shoulder press',
    'side_lateral_raise':              'lateral raise',
    'dumbbell_scaptions':              'lateral raise',
    
    # Un-filtered Composite multi-phase kinematics for Sequential State Machine tracking:
    'man_maker':                       'burpee_variant',
    'clean_and_press':                 'clean_and_press',
    'burpees':                         'burpee',
    'overhead_extension_thruster':     'thruster',
    
    # REMOVED due to Alternating Unilateral mechanics:
    # - 'dumbbell_reverse_lunge' (Angle extractor selects only one leg)
}

print(f"Mapped {len(FIT3D_TO_CONFIG_NAME)} Fit3D actions to exercise configs.")


Mapped 14 Fit3D actions to exercise configs.


## 10.2 Angle Extraction from joints3d_25

In [23]:
def calculate_angle_3d(a, b, c):
    """Computes the angle at joint B (in degrees) between vectors BA and BC."""
    ba = a - b
    bc = c - b
    n1 = np.linalg.norm(ba)
    n2 = np.linalg.norm(bc)
    if n1 < 1e-6 or n2 < 1e-6:
        return np.nan
    cosine_angle = np.dot(ba, bc) / (n1 * n2)
    return np.degrees(np.arccos(np.clip(cosine_angle, -1.0, 1.0)))


def extract_angle_sequence(joints, metric_name, metric_config):
    """Calculates the 1D angle curve for the primary joint to prevent global drift."""
    j_names = metric_config.get('joints', [])
    if len(j_names) != 3: 
        return None
    
    # Grab the left-side indices (index 0 in the FIT3D mapping tuple)
    idx_a = FIT3D_JOINT_MAP.get(j_names[0], (None,))[0]
    idx_b = FIT3D_JOINT_MAP.get(j_names[1], (None,))[0]
    idx_c = FIT3D_JOINT_MAP.get(j_names[2], (None,))[0]
    
    if None in (idx_a, idx_b, idx_c): 
        return None

    T = len(joints)
    angles = np.zeros(T)
    
    for fi in range(T):
        frame = joints[fi]
        a, b, c = frame[idx_a], frame[idx_b], frame[idx_c]
        
        if isinstance(a, np.ndarray) and isinstance(b, np.ndarray) and isinstance(c, np.ndarray):
            angles[fi] = calculate_angle_3d(a, b, c)
        else:
            angles[fi] = np.nan
            
    return angles


def get_primary_metric(action_name):
    """Returns metric details. Now supports multi-joint blending for complex exercises."""
    config_name = FIT3D_TO_CONFIG_NAME.get(action_name)
    if config_name is None or config_name not in EXERCISE_TO_CONFIG:
        return None

    ex_config = EXERCISE_TO_CONFIG[config_name]
    ref       = ex_config.get('thresholds', {})
    
    # TACTICAL OVERRIDES: Define joint blends for better stability
    overrides = {
        'deadlift':                         ['HIP_EXTENSION', 'KNEE_ANGLE'],
        'dumbbell_overhead_shoulder_press': ['SHOULDER_ABDUCTION'],
        'neutral_overhead_shoulder_press':  ['SHOULDER_ABDUCTION'],
        'diamond_pushup':                   ['ELBOW_ANGLE'] 
    }

    selected_metrics = overrides.get(action_name, [ex_config.get('metrics', [None])[0]])
    t_factor = 0.15 if action_name == 'diamond_pushup' else 0.25
    
    results = []
    for m in selected_metrics:
        mdef = JOINT_DEFINITIONS.get(m)
        if not mdef: continue
        
        # Fixed the f-string syntax error here
        rmin = ref.get(f"{m}_min")
        rmax = ref.get(f"{m}_max")
        
        # Hard-coded ROM failsafes for SHOULDER_ABDUCTION
        if (rmin is None or rmax is None) and m == 'SHOULDER_ABDUCTION':
            rmin, rmax = 80.0, 160.0
            
        if rmin is not None and rmax is not None:
            results.append((m, mdef, rmin + t_factor * (rmax - rmin), rmax - t_factor * (rmax - rmin)))

    return results if results else None


print("Angle extraction functions defined.")


Angle extraction functions defined.


## 10.3 Repetition Boundary Detection

In [ ]:
from scipy.signal import savgol_filter
import numpy as np

def detect_rep_boundaries(angles, rep_low, rep_high):
    """
    Hybrid AIFit + Heuristic segmentation.
    Uses hysteresis to ignore untrimmed video tails, and Continuous Domain Relaxation
    to snap boundaries to local extrema for maximum IoU.
    """
    valid_mask = ~np.isnan(angles)
    if valid_mask.sum() < 10:
        return [] # Prevent false positive counts on dead sequences

    interp_angles = angles.copy()
    if not valid_mask.all():
        interp_angles = np.interp(
            np.arange(len(angles)),
            np.where(valid_mask)[0],
            angles[valid_mask]
        )

    win = min(11, int(valid_mask.sum() / 4) * 2 + 1)
    win = max(win, 3)
    try:
        smoothed = savgol_filter(interp_angles, win, 2)
    except Exception:
        smoothed = interp_angles

    # 1. Dual-Hysteresis State Machine (Immune to standing around/untrimmed tails)
    state = 0 if smoothed[0] > rep_high else 1
    toggles = []

    for i in range(1, len(smoothed)):
        val = smoothed[i]
        if state == 0 and val < rep_low:
            state = 1
            toggles.append(i)
        elif state == 1 and val > rep_high:
            state = 0
            toggles.append(i)

    if len(toggles) < 2:
        return []

    # 2. Extract completed repetitions (A full rep requires 2 state toggles)
    rough_boundaries = []
    for i in range(1, len(toggles), 2):
        rough_boundaries.append(toggles[i])

    # 3. Continuous Domain Relaxation: Snap to the true local extrema (Peaks) to maximize IoU
    refined_boundaries = [0]
    
    # Find the true start of the first rep (search backwards up to 40 frames from the first drop)
    search_start = max(0, toggles[0] - 40)
    refined_boundaries[0] = search_start + np.argmax(smoothed[search_start:toggles[0]+1])

    # Snap the end of each rep to the maximum extension point (the standing/resting position)
    for b in rough_boundaries:
        search_end = min(len(smoothed) - 1, b + 40)
        local_peak = b + np.argmax(smoothed[b:search_end+1])
        refined_boundaries.append(local_peak)

    return sorted(list(set(refined_boundaries)))
print("Repetition boundary detector defined.")

Repetition boundary detector defined.


## 10.4 Metric Functions (IoU, OBO, MAE)

In [25]:
def intervals_from_bounds(bounds):
    """Convert boundary list [b0, b1, ..., bn] into (len-1) intervals."""
    return [(bounds[i], bounds[i + 1]) for i in range(len(bounds) - 1)]


def iou_interval(a, b):
    """IoU between two 1-D intervals a=(s1,e1) and b=(s2,e2)."""
    inter_start = max(a[0], b[0])
    inter_end   = min(a[1], b[1])
    if inter_end <= inter_start:
        return 0.0
    intersection = inter_end - inter_start
    union = (a[1] - a[0]) + (b[1] - b[0]) - intersection
    return intersection / union if union > 0 else 0.0


def mean_iou_sequence(gt_bounds, pred_bounds):
    """
    Compute mean IoU between GT and predicted repetition intervals.

    Each GT interval is matched to the best-IoU predicted interval (greedy).
    Unmatched GT intervals contribute IoU = 0.
    """
    gt_ivs   = intervals_from_bounds(gt_bounds)
    pred_ivs = intervals_from_bounds(pred_bounds)

    if not gt_ivs:
        return np.nan
    if not pred_ivs:
        return 0.0

    ious     = []
    used_pred = set()
    for gt_iv in gt_ivs:
        best_iou, best_j = 0.0, -1
        for j, pred_iv in enumerate(pred_ivs):
            if j in used_pred:
                continue
            s = iou_interval(gt_iv, pred_iv)
            if s > best_iou:
                best_iou, best_j = s, j
        if best_j >= 0:
            used_pred.add(best_j)
        ious.append(best_iou)

    return float(np.mean(ious))


print("Metric functions (IoU, OBO, MAE) defined.")

Metric functions (IoU, OBO, MAE) defined.


## 10.5 Full Evaluation Loop

In [ ]:
def evaluate_rep_counting(dataset_base, subj_names):
    """
    Full repetition segmentation & counting evaluation.

    Args:
        dataset_base   : root path of fit3d_data
        subj_names     : list of subject IDs to evaluate

    Returns:
        results_df : pd.DataFrame with one row per (subject, action)
        summary    : dict with overall OBO, MAE, mean IoU
    """
    rows = []
    base = Path(dataset_base)

    for subj in subj_names:
        rep_ann_path = base / 'train' / subj / 'rep_ann.json'
        if not rep_ann_path.exists():
            print(f"[WARN] No rep_ann.json for {subj}")
            continue

        with open(rep_ann_path) as f:
            rep_ann = json.load(f)

        joints_dir = base / 'train' / subj / 'joints3d_25'
        for joint_file in sorted(joints_dir.glob('*.json')):
            action = joint_file.stem

            # Need GT bounds and a metric mapping
            if action not in rep_ann:
                continue
            gt_bounds = rep_ann[action]
            if len(gt_bounds) < 2:
                continue
            gt_count = len(gt_bounds) - 1

            metrics_data = get_primary_metric(action)
            if not metrics_data:
                continue

            # Load 3-D joints
            with open(joint_file) as f:
                data = json.load(f)
            joints = (np.array(data.get('joints3d_25', list(data.values())[0]))
                      if isinstance(data, dict) else np.array(data))

            # Blend signals if multiple metrics are returned
            combined_angles = None
            total_weight = 0
            metric_names = []

            for m_name, m_config, low, high in metrics_data:
                angles = extract_angle_sequence(joints, m_name, m_config)
                if angles is None or np.isnan(angles).all(): continue
                
                # Weight by Variance (Active vs Passive mining)
                # This mimics the motion energy mining in AIFit [cite: 38, 270]
                weight = np.nanvar(angles) 
                metric_names.append(m_name)
                
                if combined_angles is None:
                    combined_angles = angles * weight
                else:
                    combined_angles += angles * weight
                total_weight += weight

            if combined_angles is None or total_weight == 0: continue
            angles = combined_angles / total_weight
            
            # Use thresholds from the primary (first) joint
            rep_low, rep_high = metrics_data[0][2], metrics_data[0][3]

            pred_bounds = detect_rep_boundaries(angles, rep_low, rep_high)
            pred_count = max(0, len(pred_bounds) - 1)

            # IoU: use interior + last boundaries as predicted rep intervals
            pred_iv_bounds = [pred_bounds[0]] + pred_bounds[1:-1] + [pred_bounds[-1]]
            miou = mean_iou_sequence(gt_bounds, pred_iv_bounds)

            abs_err = abs(gt_count - pred_count)
            obo     = int(abs_err <= 1)

            rows.append({
                'subject':     subj,
                'action':      action,
                'metric_used': " + ".join(metric_names),
                'gt_count':    gt_count,
                'pred_count':  pred_count,
                'abs_error':   abs_err,
                'obo':         obo,
                'mean_iou':    miou,
            })

    if not rows:
        print("No evaluable sequences found.")
        return pd.DataFrame(), {}

    results_df = pd.DataFrame(rows)
    summary = {
        'Total Sequences': len(results_df),
        'MAE':             results_df['abs_error'].mean(),
        'OBO Error':       1.0 - results_df['obo'].mean(),
        'Mean IoU':        results_df['mean_iou'].dropna().mean(),
    }
    return results_df, summary


print("Evaluation loop defined.")


Evaluation loop defined.


## 10.6 Run Evaluation (Validation Subjects)

In [27]:
rep_results_df, rep_summary = evaluate_rep_counting(
    dataset_Folder,
    val_subj_names
)

if not rep_results_df.empty:
    print("\n=== Per-Action Results ===")
    display(rep_results_df.sort_values(['subject', 'action']).reset_index(drop=True))

    print("\n=== Per-Exercise Summary ===")
    per_action = rep_results_df.groupby('action').agg(
        GT_Count_Mean=('gt_count',   'mean'),
        Pred_Count_Mean=('pred_count','mean'),
        MAE=('abs_error',            'mean'),
        OBO_Error=('obo',            lambda x: 1.0 - x.mean()),
        Mean_IoU=('mean_iou',        'mean'),
    ).round(3)
    display(per_action)

    print("\n=== Overall Summary ===")
    for k, v in rep_summary.items():
        print(f"  {k:25s}: {v:.4f}" if isinstance(v, float) else f"  {k:25s}: {v}")


=== Per-Action Results ===


,subject,action,metric_used,gt_count,pred_count,abs_error,obo,mean_iou
0,s03,deadlift,HIP_EXTENSION + KNEE_ANGLE,5,7,2,0,0.545905
1,s03,diamond_pushup,ELBOW_ANGLE,5,3,2,0,0.107620
2,s03,dumbbell_biceps_curls,ELBOW_ANGLE,5,5,0,1,0.932448
3,s03,dumbbell_hammer_curls,ELBOW_ANGLE,5,5,0,1,0.948534
4,s03,dumbbell_overhead_shoulder_press,SHOULDER_ABDUCTION,5,4,1,1,0.219451
5,s03,dumbbell_scaptions,SHOULDER_ABDUCTION,5,5,0,1,0.695225
6,s03,neutral_overhead_shoulder_press,SHOULDER_ABDUCTION,6,6,0,1,0.542026
7,s03,pushup,ELBOW_ANGLE,5,5,0,1,0.888853
8,s03,side_lateral_raise,SHOULDER_ABDUCTION,5,5,0,1,0.722028
9,s03,squat,KNEE_ANGLE,5,5,0,1,0.949170



=== Per-Exercise Summary ===


,GT_Count_Mean,Pred_Count_Mean,MAE,OBO_Error,Mean_IoU
action,,,,,
deadlift,5.0,6.5,1.5,0.5,0.512
diamond_pushup,5.5,4.5,1.0,0.5,0.300
dumbbell_biceps_curls,5.0,5.0,0.0,0.0,0.943
dumbbell_hammer_curls,5.0,5.0,0.0,0.0,0.943
dumbbell_overhead_shoulder_press,5.0,4.5,0.5,0.0,0.431
dumbbell_scaptions,5.0,5.0,0.0,0.0,0.686
neutral_overhead_shoulder_press,5.5,5.5,0.0,0.0,0.564
pushup,5.0,5.0,0.0,0.0,0.892
side_lateral_raise,5.0,4.5,0.5,0.0,0.624



=== Overall Summary ===
  Total Sequences          : 20
  MAE                      : 0.3500
  OBO Error                : 0.1000
  Mean IoU                 : 0.6812
